<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/stress_subgroup_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#cell 1
!pip install -q requests pandas goatools

import io, re, time
import requests
import pandas as pd

UP = 'https://rest.uniprot.org'

from google.colab import files
uploaded = files.upload()                       # peptide_gene_descriptions.csv
CSV = list(uploaded.keys())[0]

long = pd.read_csv(CSV)

# rebuild query_id (the DB-specific form) from gene_id
def to_query_id(crop, g):
    g = str(g)
    if crop == 'tomato':
        g = g.split('.')[0]
        m = re.match(r'^SOLYC(\d+)G(\d+)$', g)
        return f'Solyc{m.group(1)}g{m.group(2)}' if m else g
    if crop == 'cotton':
        return g.replace('LOC', '')
    return g

long['query_id'] = [to_query_id(c, g) for c, g in zip(long.crop, long.gene_id)]

print(f'{len(long)} rows | {long.gene_id.nunique()} unique genes')
print(long.groupby('crop').query_id.nunique().to_string())

In [ ]:
#cell 2
import json, os

CACHE = 'go_cache'
os.makedirs(CACHE, exist_ok=True)

ORGANISM = {'soy': 3847, 'chilli': 4072, 'tomato': 4081}   # cotton + grape handled separately


def parse_go(cell):
    """UniProt go_id field -> set of GO IDs. Handles 'GO:0006979; GO:0009408'."""
    if not isinstance(cell, str) or not cell.strip():
        return []
    return sorted(set(re.findall(r'GO:\d{7}', cell)))


# ---- soy: one request per gene (GLYMA IDs don't appear in UniProt output cols) ----
def fetch_soy(ids, org=3847):
    rows = []
    for n, gid in enumerate(ids, 1):
        for attempt in range(3):
            try:
                r = requests.get(f'{UP}/uniprotkb/search',
                                 params={'query': f'{gid} AND organism_id:{org}',
                                         'format': 'tsv',
                                         'fields': 'protein_name,go_id',
                                         'size': 1}, timeout=60)
                r.raise_for_status()
                b = r.text.strip().split('\n')
                if len(b) > 1:
                    p = b[1].split('\t')
                    rows.append({'query_id': gid,
                                 'description': p[0] if len(p) > 0 else '',
                                 'go_ids': parse_go(p[1] if len(p) > 1 else '')})
                break
            except Exception:
                if attempt == 2:
                    rows.append({'query_id': gid, 'description': '', 'go_ids': []})
                time.sleep(3)
        if n % 25 == 0 or n == len(ids):
            print(f'    soy {n}/{len(ids)}', end='\r')
        time.sleep(0.1)
    print()
    return rows


# ---- chilli / tomato: batched search, then back-map GO to the matching ID ----
def fetch_batched(ids, org, batch=40):
    rows = {}
    for i in range(0, len(ids), batch):
        chunk = ids[i:i + batch]
        q = '(' + ' OR '.join(chunk) + f') AND organism_id:{org}'
        for attempt in range(3):
            try:
                r = requests.get(f'{UP}/uniprotkb/search',
                                 params={'query': q, 'format': 'tsv',
                                         'fields': 'protein_name,go_id,xref_ensemblplants,gene_names',
                                         'size': 500}, timeout=120)
                r.raise_for_status()
                t = pd.read_csv(io.StringIO(r.text), sep='\t').fillna('')
                for _, row in t.iterrows():
                    text = ' '.join(row.astype(str)).upper()
                    for gid in chunk:
                        if gid.upper() in text and gid not in rows:
                            rows[gid] = {'query_id': gid,
                                         'description': row.get('Protein names', ''),
                                         'go_ids': parse_go(str(row.get('Gene Ontology IDs', '')))}
                break
            except Exception:
                if attempt == 2:
                    pass
                time.sleep(5)
        print(f'    {min(i+batch, len(ids))}/{len(ids)}', end='\r')
    print()
    return list(rows.values())


# ---- cotton: idmapping (GeneID -> UniProtKB), GO in the stream ----
def fetch_cotton(ids):
    job = requests.post(f'{UP}/idmapping/run',
                        data={'from': 'GeneID', 'to': 'UniProtKB',
                              'ids': ','.join(ids)}, timeout=60)
    job.raise_for_status()
    jid = job.json()['jobId']
    for _ in range(60):
        s = requests.get(f'{UP}/idmapping/status/{jid}', timeout=60).json()
        if s.get('jobStatus') not in ('RUNNING', 'NEW'):
            break
        time.sleep(3)
    r = requests.get(f'{UP}/idmapping/uniprotkb/results/stream/{jid}',
                     params={'format': 'tsv', 'fields': 'protein_name,go_id'}, timeout=300)
    r.raise_for_status()
    t = pd.read_csv(io.StringIO(r.text), sep='\t').fillna('')
    rows = {}
    for _, row in t.iterrows():
        gid = str(row['From'])
        if gid not in rows:
            rows[gid] = {'query_id': gid,
                         'description': row.get('Protein names', ''),
                         'go_ids': parse_go(str(row.get('Gene Ontology IDs', '')))}
    return list(rows.values())


def run(crop, fn, *a):
    path = f'{CACHE}/{crop}.json'
    if os.path.exists(path):
        data = json.load(open(path))
        print(f'[{crop}] cached: {len(data)} genes')
        return data
    ids = sorted(long.loc[long.crop == crop, 'query_id'].unique())
    print(f'[{crop}] fetching {len(ids)} genes...')
    data = fn(ids, *a)
    json.dump(data, open(path, 'w'))
    withgo = sum(1 for d in data if d['go_ids'])
    print(f'[{crop}] done: {len(data)} genes, {withgo} with GO')
    return data


go_soy    = run('soy',    fetch_soy)
go_chilli = run('chilli', lambda ids: fetch_batched(ids, 4072))
go_tomato = run('tomato', lambda ids: fetch_batched(ids, 4081))
go_cotton = run('cotton', fetch_cotton)

print('\ndone — grape comes from the Blast2GO file next')

In [ ]:
#cell 3
import os
from google.colab import files

TSV = 'PN40024.v4.1.REF.b2g.tsv'

# upload only if it's not already in the session
if not os.path.exists(TSV):
    print(f'{TSV} not found — upload it (the .tsv, or the .zip and unzip first)')
    up = files.upload()
    name = list(up.keys())[0]
    if name.endswith('.zip'):
        import zipfile
        with zipfile.ZipFile(name) as z:
            z.extractall('.')
            print('unzipped:', z.namelist())
        # the tsv is inside; find it
        TSV = next(f for f in z.namelist() if f.endswith('.tsv'))
    else:
        TSV = name

print('using:', TSV)

b2g = pd.read_csv(TSV, sep='\t',
                  usecols=['Sequence Name', 'Sequence Description', 'Annotation GO ID'])

b2g['query_id'] = (b2g['Sequence Name']
                   .str.replace(r'_t\d+$', '', regex=True)
                   .str.upper())
b2g['description'] = (b2g['Sequence Description'].astype(str)
                      .str.replace(r'^[A-Z]{2,3}[_]?\d{5,}\.\d+', '', regex=True)
                      .str.strip())
b2g['go_ids'] = b2g['Annotation GO ID'].apply(parse_go)

b2g = b2g[['query_id', 'description', 'go_ids']].drop_duplicates('query_id')

mine = set(long.loc[long.crop == 'grape', 'query_id'])
go_grape = b2g[b2g.query_id.isin(mine)].to_dict('records')

withgo = sum(1 for d in go_grape if d['go_ids'])
print(f'[grape] {len(go_grape)}/{len(mine)} genes, {withgo} with GO')

In [ ]:
#cell 4

import os
from goatools.obo_parser import GODag

# --- download go-basic.obo (Colab has full internet) ---
if not os.path.exists('go-basic.obo'):
    !wget -q http://purl.obolibrary.org/obo/go/go-basic.obo
godag = GODag('go-basic.obo')

# --- target buckets: named children of GO:0006950 ---
BUCKETS = {
    'oxidative':        'GO:0006979',   # response to oxidative stress
    'heat':             'GO:0009408',   # response to heat
    'cold':             'GO:0009409',   # response to cold
    'water_deprivation':'GO:0009414',   # drought
    'osmotic':          'GO:0006970',   # response to osmotic stress
    'salt':             'GO:0009651',   # response to salt stress
    'hypoxia':          'GO:0001666',   # response to hypoxia
    'starvation':       'GO:0042594',   # response to starvation
    'wounding':         'GO:0009611',   # response to wounding
    'dna_damage':       'GO:0006974',   # DNA damage response
    'er_stress':        'GO:0034976',   # ER / unfolded protein
    'defense':          'GO:0006952',   # defense response (biotic)
}

# expand each bucket to itself + all descendants
bucket_sets = {}
for name, gid in BUCKETS.items():
    if gid in godag:
        bucket_sets[name] = {gid} | godag[gid].get_all_children()
    else:
        print(f'  ! {gid} ({name}) not in OBO')

stress_all = {'GO:0006950'} | godag['GO:0006950'].get_all_children()

# --- assemble every gene's GO set, tagged with crop ---
records = []
for crop, data in [('soy', go_soy), ('chilli', go_chilli), ('tomato', go_tomato),
                   ('cotton', go_cotton), ('grape', go_grape)]:
    for d in data:
        records.append({'crop': crop,
                        'query_id': d['query_id'],
                        'description': d.get('description', ''),
                        'go_ids': set(d['go_ids'])})
G = pd.DataFrame(records)

# --- label each gene ---
def label(go_set):
    return [name for name, s in bucket_sets.items() if go_set & s]

G['buckets'] = G.go_ids.apply(label)
G['in_stress'] = G.go_ids.apply(lambda s: bool(s & stress_all))
G['n_buckets'] = G.buckets.apply(len)

# a gene under stress but in none of the named children
G['other_stress'] = G.in_stress & (G.n_buckets == 0)

print('genes per bucket (a gene can be in several):')
counts = pd.Series([b for bl in G.buckets for b in bl]).value_counts()
print(counts.to_string())
print(f'\nother_stress (under 0006950 but no named bucket): {G.other_stress.sum()}')
print(f'no stress GO / no GO at all: {(~G.in_stress).sum()}')

In [ ]:
missed = G[~G.in_stress].copy()
print(f'{len(missed)} genes not under GO:0006950\n')

# how many have GO at all vs none
missed['has_go'] = missed.go_ids.apply(lambda s: len(s) > 0)
print(f'  {(~missed.has_go).sum()} have NO GO at all')
print(f'  {missed.has_go.sum()} have GO, but none under stress\n')

# keyword-sniff the descriptions: are these secretly stress genes UniProt under-annotated?
STRESS_KW = (r'heat shock|chaperon|\bHSP\b|dnaj|peroxidas|catalase|superoxide|'
             r'glutathione|thioredoxin|oxidative|dehydrin|\bLEA\b|late embryogenesis|'
             r'desiccation|drought|osmotic|salt|dna repair|helicase|recombinat|'
             r'excision repair|pathogenesis|defensin|disease resist|\bNBS-LRR\b|'
             r'wound|senescence|abscisic|\bABA\b|cold|freezing|aquaporin')
missed['looks_stress'] = missed.description.str.contains(STRESS_KW, case=False, na=False)

n_looks = missed.looks_stress.sum()
print(f'{n_looks}/{len(missed)} have stress-suggestive descriptions '
      f'(under-annotated in GO, not truly off-target)\n')

print('--- sample that LOOK like stress genes (GO just missing) ---')
print(missed[missed.looks_stress][['crop','query_id','description']]
      .head(15).to_string(index=False))

print('\n--- sample that do NOT look stress-related ---')
print(missed[~missed.looks_stress][['crop','query_id','description']]
      .head(15).to_string(index=False))

print('\nby crop:')
print(missed.groupby('crop').size().to_string())

missed_out = missed[['crop','query_id','description','has_go','looks_stress']].copy()
missed_out.to_csv('genes_no_stress_go.csv', index=False)
from google.colab import files
files.download('genes_no_stress_go.csv')

In [ ]:
missed = G[~G.in_stress].copy()

# curated stress-regulator families (specific, not generic "kinase"/"TF")
REGULATOR_FAMILIES = {
    'WRKY':        r'WRKY',
    'AP2/ERF/DREB':r'AP2|ethylene.responsive|ERF\b|DREB|CBF\b',
    'NAC':         r'\bNAC\b|NAM\b|no apical meristem',
    'MYB':         r'\bMYB\b',
    'bZIP/ABF':    r'bZIP|ABF\b|ABRE|AREB',
    'HSF':         r'heat stress transcription|heat shock factor|\bHSF\b',
    'MAPK':        r'MAP kinase|MAPK|mitogen.activated',
    'CDPK/CIPK':   r'calcium.dependent protein kinase|CDPK|CIPK|CBL.interacting',
    'ABA core':    r'PYR/PYL|PYL\b|abscisic acid receptor|SnRK2|SNF1.related|PP2C|'
                   r'protein phosphatase 2C|ABI\b|ABA.insensitive',
    'Ca signalling':r'calmodulin|calcineurin B',
}

def reg_family(desc):
    d = str(desc)
    hits = [name for name, pat in REGULATOR_FAMILIES.items()
            if re.search(pat, d, flags=re.I)]
    return ';'.join(hits)

missed['regulator_family'] = missed.description.apply(reg_family)
missed['tier'] = missed.regulator_family.apply(
    lambda f: 'regulator' if f else 'unresolved')

# assemble the 3-tier view
G['tier'] = 'effector'
G.loc[~G.in_stress, 'tier'] = missed['tier']
G.loc[~G.in_stress, 'regulator_family'] = missed['regulator_family']

print('tier breakdown:')
print(G.tier.value_counts().to_string())
print(f'\nrescued regulators: {(G.tier == "regulator").sum()}')
print('\nregulator families found:')
fam = pd.Series([f for fs in missed.regulator_family if fs for f in fs.split(';')])
print(fam.value_counts().to_string())

print('\n--- sample rescued regulators ---')
print(G[G.tier == 'regulator'][['crop','description','regulator_family']]
      .head(20).to_string(index=False))

In [ ]:
# per-gene, with tier + bucket + regulator family
gene_final = G[['crop', 'query_id', 'description', 'tier',
                'buckets', 'regulator_family', 'in_stress']].copy()
gene_final['buckets'] = gene_final.buckets.apply(lambda b: ';'.join(b) if b else '')
gene_final['regulator_family'] = gene_final.regulator_family.fillna('')
gene_final.to_csv('genes_tiered.csv', index=False)

# join tier + buckets back onto peptides
key = G.set_index(['crop', 'query_id'])[['tier', 'buckets', 'regulator_family']]
def lookup(row, col):
    try: return key.loc[(row.crop, row.query_id), col]
    except KeyError: return None
long['tier'] = long.apply(lambda r: lookup(r, 'tier'), axis=1)
long['buckets'] = long.apply(lambda r: lookup(r, 'buckets'), axis=1)
long['bucket_str'] = long.buckets.apply(lambda b: ';'.join(b) if b else '')
long.drop(columns=['buckets']).to_csv('peptides_tiered.csv', index=False)

# reportable set = effector + regulator
report = G[G.tier.isin(['effector', 'regulator'])]
print(f'reportable stress set: {len(report)} genes '
      f'({(G.tier=="effector").sum()} effector + {(G.tier=="regulator").sum()} regulator)')
print(f'flagged unresolved:    {(G.tier=="unresolved").sum()}\n')

# per-peptide dominant axis, reportable set only
pep = (long[long.tier.isin(['effector','regulator']) & (long.bucket_str != '')]
       .assign(bucket=long.bucket_str.str.split(';')).explode('bucket'))
if len(pep):
    dom = (pep.groupby(['peptide_id','bucket'])
              .agg(n_crops=('crop','nunique')).reset_index()
              .sort_values('n_crops', ascending=False)
              .drop_duplicates('peptide_id'))
    print('peptides by dominant stress axis:')
    print(dom.bucket.value_counts().to_string())

from google.colab import files
files.download('genes_tiered.csv')
files.download('peptides_tiered.csv')

In [ ]:
orig = (long[['crop', 'query_id', 'gene_id']]
        .drop_duplicates(['crop', 'query_id']))
G = G.merge(orig, on=['crop', 'query_id'], how='left')

In [ ]:
BUCKET_GO = {
    'oxidative':'GO:0006979','heat':'GO:0009408','cold':'GO:0009409',
    'water_deprivation':'GO:0009414','osmotic':'GO:0006970','salt':'GO:0009651',
    'hypoxia':'GO:0001666','starvation':'GO:0042594','wounding':'GO:0009611',
    'dna_damage':'GO:0006974','er_stress':'GO:0034976','defense':'GO:0006952',
}

gb = G[G.buckets.apply(len) > 0][['crop','query_id','description','tier','buckets']].copy()

# join peptides, then explode so each (gene, bucket) is its own group
out = (long[['crop','query_id','gene_id','peptide_id','peptide_seq']]
       .merge(gb, on=['crop','query_id'], how='inner')
       .explode('buckets')
       .rename(columns={'buckets':'bucket'}))
out['bucket_go'] = out.bucket.map(BUCKET_GO)

out = (out.sort_values(['bucket_go','crop','gene_id','peptide_id'])
          [['bucket_go','bucket','crop','gene_id','description',
            'tier','peptide_id','peptide_seq']]
          .reset_index(drop=True))

print(f'{len(out)} rows, grouped by GO bucket\n')
print(out.groupby('bucket_go').size().to_string())
print()
print(out.head(30).to_string(index=False))

out.to_csv('peptide_gene_by_bucket_go.csv', index=False)
from google.colab import files
files.download('peptide_gene_by_bucket_go.csv')

In [ ]:
BUCKET_GO = {
    'oxidative':'GO:0006979','heat':'GO:0009408','cold':'GO:0009409',
    'water_deprivation':'GO:0009414','osmotic':'GO:0006970','salt':'GO:0009651',
    'hypoxia':'GO:0001666','starvation':'GO:0042594','wounding':'GO:0009611',
    'dna_damage':'GO:0006974','er_stress':'GO:0034976','defense':'GO:0006952',
}

gb = G[G.buckets.apply(len) > 0][['crop','query_id','description','tier','buckets']].copy()

out = (long[['crop','query_id','gene_id','peptide_id','peptide_seq']]
       .merge(gb, on=['crop','query_id'], how='inner')
       .explode('buckets')
       .rename(columns={'buckets':'bucket'}))
out['bucket_go'] = out.bucket.map(BUCKET_GO)

# sort: GO id -> peptide -> gene, so shared peptides cluster within each GO
out = (out.sort_values(['bucket_go','peptide_seq','peptide_id','gene_id'])
          [['bucket_go','bucket','peptide_seq','peptide_id',
            'crop','gene_id','description','tier']]
          .reset_index(drop=True))

print(f'{len(out)} rows\n')

# show peptides hitting >=2 genes within the same GO bucket (the pattern you're after)
multi = (out.groupby(['bucket_go','peptide_seq'])
            .agg(n_genes=('gene_id','nunique'),
                 n_crops=('crop','nunique')).reset_index()
            .query('n_genes >= 2')
            .sort_values(['bucket_go','n_genes'], ascending=[True,False]))
print('peptides hitting multiple genes within one GO bucket:')
print(multi.head(25).to_string(index=False))

print('\n--- full table head ---')
print(out.head(30).to_string(index=False))

out.to_csv('peptide_gene_by_go_peptide.csv', index=False)
from google.colab import files
files.download('peptide_gene_by_go_peptide.csv')

In [ ]:
print(G.tier.value_counts().to_string())
print()
print('effectors with a bucket:', G[(G.tier=='effector') & (G.buckets.apply(len)>0)].shape[0])
print('regulators with a bucket:', G[(G.tier=='regulator') & (G.buckets.apply(len)>0)].shape[0])

In [ ]:
BUCKET_GO = {
    'oxidative':'GO:0006979','heat':'GO:0009408','cold':'GO:0009409',
    'water_deprivation':'GO:0009414','osmotic':'GO:0006970','salt':'GO:0009651',
    'hypoxia':'GO:0001666','starvation':'GO:0042594','wounding':'GO:0009611',
    'dna_damage':'GO:0006974','er_stress':'GO:0034976','defense':'GO:0006952',
}

# effectors with a named bucket -> one row per (gene, bucket)
eff = G[G.buckets.apply(len) > 0][['crop','query_id','description','tier','buckets']].copy()
eff = eff.explode('buckets').rename(columns={'buckets':'bucket'})
eff['bucket_go'] = eff.bucket.map(BUCKET_GO)

# regulators -> synthetic group, family in place of bucket
reg = G[G.tier=='regulator'][['crop','query_id','description','tier','regulator_family']].copy()
reg['bucket']    = 'regulator:' + reg.regulator_family
reg['bucket_go'] = 'ZZ_regulator'          # sorts last

both = pd.concat([eff, reg[['crop','query_id','description','tier','bucket','bucket_go']]],
                 ignore_index=True)

out = (long[['crop','query_id','gene_id','peptide_id','peptide_seq']]
       .merge(both, on=['crop','query_id'], how='inner')
       .sort_values(['bucket_go','peptide_seq','peptide_id','gene_id'])
       [['bucket_go','bucket','tier','peptide_seq','peptide_id',
         'crop','gene_id','description']]
       .reset_index(drop=True))

print(out.tier.value_counts().to_string(), '\n')
print(out.groupby('bucket_go').size().to_string())

out.to_csv('peptide_gene_effector_regulator.csv', index=False)
from google.colab import files
files.download('peptide_gene_effector_regulator.csv')

In [ ]:
sig = (out.groupby(['bucket_go','peptide_seq'])
          .agg(n_genes=('gene_id','nunique'),
               n_crops=('crop','nunique'),
               tier=('tier','first')).reset_index()
          .query('n_genes >= 3')
          .sort_values(['n_genes','n_crops'], ascending=False))
print(f'{len(sig)} peptide×bucket combos hitting >=3 genes\n')
print(sig.head(25).to_string(index=False))